In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
sys.path.extend(['../../bgc-analysis','C:/Users/athay/OneDrive/Documents/Repositories/test-analysis','C:/Users/athay/OneDrive/Documents/Repositories/flowthrough_testing'])
from flowthrough_analysis import diff_analysis
OUTPUT_ROOT = Path(r'Z:/www/bgc/flow_through')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PARAM_LABELS = {'MCOMS3':'Backscattering at 700nm (m^-1)','MCOMS3-Median':'Backscattering at 700nm (m^-1) - Median','PH_TOTAL':'PH (total)','PH_TOTAL-Median':'PH (total) - Median','MCOMS2':'Chlorophyll a (mg/m^3)','MCOMS2-Median':'Chlorophyll a (mg/m^3) - Median','MCOMS1':'FDOM (ppb)','MCOMS1-Median':'FDOM (ppb) - Median','DOXY':'Dissolved Oxygen (umol/kg)','DOXY-Median':'Dissolved Oxygen (umol/kg) - Median','PSAL':'Salinity (psu)','PSAL-Median':'Salinity - Median (psu)','TEMP':'Temperature (C)','TEMP-Median':'Temperature (C) - Median','NITRATE':'Nitrate (uM)','NITRATE-Median':'Nitrate (uM) - Median','PRES':'Pressure (dbar)','PRES-Median':'Pressure (dbar) - Median'}
_analysis_cache = {}
def get_analysis(sensor, parameter):
    key = (str(sensor), parameter)
    if key not in _analysis_cache:
        _analysis_cache[key] = diff_analysis(*key)
    return _analysis_cache[key]
def combined_data(sensors, parameter):
    return pd.concat((get_analysis(sensor, parameter) for sensor in sensors), ignore_index=True)

In [2]:
SN = ['1558','1557','1559','1554','1540','1541','1483','1358','1528','1529','1530','1539','1560','1561','1562','1556','1542','1531','1565','1566','1583','1633','1634','1635','1533','1567','1568','1569','1573','1574','1481','1577','1581','1636','1637','1638','1639','1564','1640','1641','1642','1643','1481','1661','1652','1555','1662','1663','1563','1580','1664','1659','1664','1669','1673','1665','1757','1759','1760','2002','2003','2001','2004','1668','1541','2007','2008','2005','2009','2010','2011','2012','1554','1754','1658','2014','2015','2016','2018','1660','1755','1758']
param = ['PSAL','PH_TOTAL','TEMP','PRES','DOXY','MCOMS1','MCOMS2','MCOMS3','NITRATE']
params = ['PSAL','PH_TOTAL','TEMP','DOXY','NITRATE','PRES']

In [3]:
for parameter in param:
    df = combined_data(SN, parameter)
    fig = px.scatter(df, x='Time Since Test Start (hours)', y=f'{parameter}-Median', color='Float Number', labels=PARAM_LABELS, title='Time Series Plot')
    fig.write_html(OUTPUT_ROOT / f'{parameter}.html')

In [4]:
for parameter in params:
    df = combined_data(SN, parameter)
    fig = px.scatter(df, x='Time Since Test Start (hours)', y=parameter, color='Float Number', labels=PARAM_LABELS, title='Time Series Plot')
    fig.write_html(OUTPUT_ROOT / f'{parameter}.html')

In [5]:
groups = [[1558,1557,1559,1554,1540,1541],[1483,1358,1528,1529,1530,1539],[1560,1561,1562,1556],[1542,1531],[1565,1566,1583,1633,1634,1635],[1533,1567,1568,1569,1573,1574],[1636,1637,1638,1481,1581,1577],[1639,1564,1640,1641,1642,1643],[1563,1662,1663],[1555,1661,1652],[1664,1580,1659],[1664,1669,1673],[1757,1759,1665],[1760,2002,2003],[2001,2004],[1668,1541],[2007,2008,2005],[2009,2010,2011,2012],[1554,1754,1658],[2014,2015,2016,2018],[1758,1660,1755]]

In [6]:
for group in groups:
    for parameter in param:
        fig = go.Figure()
        first_date = None
        for sensor in group:
            data = get_analysis(sensor, parameter)
            if first_date is None:
                first_date = data['DATE'].iloc[0]
            fig.add_trace(go.Scatter(x=data['DATE'], y=data[parameter], mode='markers', name=str(sensor)))
        fig.update_layout(title=f'{parameter}_{first_date}')
        output_dir = OUTPUT_ROOT / parameter
        output_dir.mkdir(parents=True, exist_ok=True)
        fig.write_html(output_dir / f'{first_date:%Y-%m-%d}.html')

c:\Users\athay\anaconda3\Lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result

